## ***初始化相关***

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import itertools
import copy
import math

from sympy.physics.quantum.gate import normalized
from tqdm import tqdm
import pandas as pd

import torch
from torch.utils.data import DataLoader
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torchvision.transforms import v2
# 防止内核挂掉
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

: 

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using Device: ',device)
BATCH_SIZE = 48
IMG_SIZE = [1000, 1000]
N_pixels = 1024
PhaseMask = [1200, 1200]
PIXEL_SIZE = 8e-6
wl = 532e-9
PADDINGx = (PhaseMask[0] - IMG_SIZE[0]) // 4  # 避免边缘信息丢失
PADDINGy = (PhaseMask[1] - IMG_SIZE[1]) // 4  # 避免边缘信息丢失


In [ ]:

# 数据预处理并加载
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
    transforms.Resize((IMG_SIZE[0], IMG_SIZE[1]), antialias=True),
    transforms.RandomRotation(degrees=1),
    #transforms.Pad([PADDINGx, PADDINGx, PADDINGy, PADDINGy], fill=(0), padding_mode='constant'),
    #transforms.Normalize((0.1307,), (0.3081,))
    transforms.RandomAffine(
        degrees=0,  # 不旋转
        translate=(0.03, 0.03),
        scale=(0.97, 1.03),  # 不缩放
        shear=None   # 不c剪切
    ),
    transforms.ElasticTransform(alpha=100.0, sigma=5.0),
    #transforms.Pad([PADDINGx, PADDINGx, PADDINGy, PADDINGy], padding_mode="edge"),
    transforms.Pad([PADDINGx, PADDINGx, PADDINGy, PADDINGy]),
    transforms.ColorJitter(brightness=0.4, contrast=0.4),
    transforms.RandomPerspective(distortion_scale=0.2, p=0.3),
    transforms.RandomAdjustSharpness(sharpness_factor=2.0, p=0.5),
])
folder = "data_pairs"
type=None
dataset="continue5"
if type!=None:
    dataset_name = type + "/" + dataset
else:
    dataset_name = dataset
num_workers = min(os.cpu_count(), 8) 
print(f"Using {num_workers} workers for data loading.")



# --- 1. CPU 预处理 (仅做最小化处理) ---
# 这一步仍在 Dataset 中运行，目的是为了能把图片转换成 Tensor 并 stack 成一个 Batch
cpu_transform = v2.Compose([
    v2.ToImage(),  # v2 新写法，替代 ToTensor
    v2.Resize((IMG_SIZE[0], IMG_SIZE[1]), antialias=True),
    v2.ToDtype(torch.float32, scale=True), # 归一化到 [0, 1] 并转 float32
    v2.Grayscale(num_output_channels=1),
])

# --- 2. GPU 增强 (耗时操作全放这里) ---
# 这是一个 nn.Module，所以可以像网络层一样直接调用，且支持 GPU
gpu_transform = v2.Compose([
    # 这里输入的是 GPU 上的 Batch [B, C, H, W]，速度极快
    v2.RandomRotation(degrees=1),
    v2.RandomAffine(
        degrees=0, 
        translate=(0.03, 0.03),
        scale=(0.97, 1.03), 
        shear=None
    ),
    v2.ElasticTransform(alpha=100.0, sigma=5.0), # 在 GPU 上运行快很多
    v2.Pad([PADDINGx, PADDINGx, PADDINGy, PADDINGy]),
    v2.ColorJitter(brightness=0.4, contrast=0.4),
    v2.RandomPerspective(distortion_scale=0.2, p=0.3),
    v2.RandomAdjustSharpness(sharpness_factor=2.0, p=0.5),
])
# Dataset 只使用 cpu_transform
train_dataset = torchvision.datasets.ImageFolder(f"{dataset_name}/train", transform=cpu_transform)
val_dataset   = torchvision.datasets.ImageFolder(f"{dataset_name}/val",   transform=cpu_transform)

# DataLoader 依然需要 num_workers
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True,persistent_workers=True)
val_dataloader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

# 定义一个绘图函数
def image_plot(image_phase, label):
    norm = np.absolute(image_phase.cpu().numpy())
    fig, (ax1, ax2) = plt.subplots(1, 2)
    ax1.imshow(np.round(norm, 5), cmap='RdBu') # 显示图片每个像素点的振幅
    ax1.axis('off')
    ax1.set_title('Amplitude')
    ax2.imshow(np.angle(image_phase.cpu().numpy()), cmap='RdBu')
    ax2.axis('off')  # 不显示坐标轴
    ax2.set_title('Phase')
    fig.suptitle("label={}".format(label), x=0.5, y=0.8)
    plt.show()

"""images_padded = None
images_E = None
images_phase = None"""
"""def load_data(dataloader, is_train=True):
    total = len(dataloader)
    mode_name = "train" if is_train else "val"
    
    for i, (images, labels) in enumerate(dataloader):
        # 1. 搬运到 GPU (异步)
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        
        # 2. 【关键】在 GPU 上执行增强
        # 无论 Train 还是 Val，都执行这一步，模拟真实环境的恶劣情况
        with torch.no_grad(): # 增强操作本身不需要记录梯度
            images_aug = gpu_transform(images)
        
        # 3. 物理转换逻辑 (保持你的原有逻辑)
        # 注意：v2 输出通常保持维度 [B, C, H, W]，如果 Pad 需要 [B, H, W] 输入需 squeeze
        # 但 F.pad 其实支持 4D 输入，建议直接对 images_aug 操作
        if images_aug.shape[1] == 1:
             images_squeezed = images_aug.squeeze(1)
        else:
             images_squeezed = images_aug
             
        images_padded = F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy)) # 二次Pad? 代码逻辑里似乎有两次
        images_E = torch.sqrt(images_padded)
        images_phase = torch.exp(1j * 2.0 * torch.pi * images_E)
        # 打印进度
        print(f"\rbatch_number [{i+1}]/[{total}]",end='')
        if (i + 1) % 20 == 0:
            classes = torch.unique(labels).cpu().numpy()
            classes_num = len(classes)
            print('classes of the first batch: {}, number of classes: {}'.format(classes, classes_num))#  第一个batch的总类
            image_plot(images_phase[0], labels[0])

    images_E = torch.sqrt(images_padded)
    images_phase = torch.exp(1j*2.0*torch.pi*images_E) # 转换为相位图像
    labels = labels.to(device)"""


## ***Diffractive Layer***

In [ ]:
class Diffractive_Layer(torch.nn.Module):
    # 模型初始化（构造实例），默认实参波长为532e-9，网格总数50，网格大小2e-6，z方向传播0.002。
    def __init__(self, wl = wl, PhaseMask = PhaseMask, pixel_size = PIXEL_SIZE, distance = 0.15):
        super(Diffractive_Layer, self).__init__() # 初始化父类

        # 以1/d为单位频率，得到一系列频率分量[0, 1, 2, ···, N_pixels/2-1,-N_pixels/2, ···, -1]/(N_pixels*d)。
        fx = torch.fft.fftshift(torch.fft.fftfreq(PhaseMask[0], d = pixel_size)).float()
        fy = torch.fft.fftshift(torch.fft.fftfreq(PhaseMask[1], d = pixel_size)).float()
        fxx, fyy = torch.meshgrid(fx, fy) # 拉网格，每个网格坐标点为空间频率各分量。

        argument = (2 * np.pi)**2 * ((1. / wl) ** 2 - fxx ** 2 - fyy ** 2)

        # 计算传播场或倏逝场的模式kz，传播场kz为实数，倏逝场kz为复数
        tmp = torch.sqrt(torch.abs(argument))
        kz = torch.tensor(torch.where(argument >= 0, tmp, 1j*tmp)).to(device)
        self.phase = torch.exp(1j * kz * distance).to(device)
        self.phase = torch.fft.fftshift(self.phase)
        
    def forward(self, E):
        # 定义单个衍射层内的前向传播
        E = E.to(torch.cfloat)
        fft_c = torch.fft.fft2(E) # 对电场E进行二维傅里叶变换
        #c = torch.fft.fftshift(fft_c) # 将零频移至张量中心
        c = fft_c
        angular_spectrum = torch.fft.ifft2(c * self.phase) # 卷积后逆变换得到响应的角谱
        return angular_spectrum

In [ ]:
# 看一个样本经过衍射层后变成啥样
"""model = Diffractive_Layer()
E = torch.sqrt(images_padded[0])
out = model.forward(E)
image_plot(out, labels[0])"""

## ***Propagation Layer***

In [ ]:
class Propagation_Layer(torch.nn.Module):
    # 与上面衍射层大致相同，区别在于传输层是最后一个衍射层到探测器层间的部分，中间可以自定义加额外的器件。
    def __init__(self, wl = wl, PhaseMask = PhaseMask, pixel_size = PIXEL_SIZE, distance = 0.2):
        super(Propagation_Layer, self).__init__() # 初始化父类

        # 以1/d为单位频率，得到一系列频率分量[0, 1, 2, ···, N_pixels/2-1,-N_pixels/2, ···, -1]/(N_pixels*d)。
        fx = torch.fft.fftshift(torch.fft.fftfreq(PhaseMask[0], d = pixel_size)).float()
        fy = torch.fft.fftshift(torch.fft.fftfreq(PhaseMask[1], d = pixel_size)).float()
        fxx, fyy = torch.meshgrid(fx, fy) # 拉网格，每个网格坐标点为空间频率各分量。

        argument = (2 * np.pi)**2 * ((1. / wl) ** 2 - fxx ** 2 - fyy ** 2)

        # 计算传播场或倏逝场的模式kz，传播场kz为实数，倏逝场kz为复数
        tmp = np.sqrt(np.abs(argument))
        kz = torch.tensor(np.where(argument >= 0, tmp, 1j*tmp))
        self.phase = torch.exp(1j * kz * distance).to(device)
        self.phase = torch.fft.fftshift(self.phase)
        
    def forward(self, E):
        # 定义单个衍射层内的前向传播
        E = E.to(torch.cfloat)
        fft_c = torch.fft.fft2(E) # 对电场E进行二维傅里叶变换
        # c = torch.fft.fftshift(fft_c) # 将零频移至张量中心
        c = fft_c
        angular_spectrum = torch.fft.ifft2(c * self.phase) # 卷积后逆变换得到响应的角谱
        return angular_spectrum

## ***Detectors Layer***

In [ ]:
# 生成一行探测器。指定探测器个数N_det，在x方向上生成齐高等间距det_step的一组探测器
# left，right，up和down分别是该行矩形探测器的四个顶点坐标。
def generate_det_row(det_size, start_pos_x, start_pos_y, det_step, N_det):
    p = []
    for i in range(N_det):
        left = start_pos_x+i*(int(det_step)+det_size)
        right = left + det_size
        up = start_pos_y
        down = start_pos_y + det_size
        p.append((up, down, left, right))
    return p

# 生成三行探测器。利用generate_det_row函数生成三行等间距矩形探测器。
def set_det_pos(det_size = 60, start_pos_x = 36, start_pos_y = 36,
                N_det_sets = [2, 0, 2], det_steps_x = [5, 11, 5], det_steps_y = 5):
    p = []
    for i in range(len(N_det_sets)):
        p.append(generate_det_row(det_size, start_pos_x, start_pos_y+i*(det_steps_y+1)*det_size, det_steps_x[i]*det_size, N_det_sets[i]))
    return list(itertools.chain.from_iterable(p))

# def set_det_pos(det_size=20, start_pos_x = 46, start_pos_y = 46):
#     p = []
#     p.append(generate_det_row(det_size, start_pos_x, start_pos_y, 2*det_size, 3))
#     p.append(generate_det_row(det_size, start_pos_x, start_pos_y+3*det_size, 1*det_size, 4))
#     p.append(generate_det_row(det_size, start_pos_x, start_pos_y+6*det_size, 2*det_size, 3))
#     return list(itertools.chain.from_iterable(p))

# 获取最终衍射光强在各个探测器上的分布情况
def detector_region(Int,detector_mask=None,detector_minus=None,detector_pos=None,detector_size=60):
    #detectors_list = []
    detectors_list=torch.zeros(Int.shape[0],len(detector_pos), device = device)
    total = None
    #full_Int = Int.sum(dim=(1,2)) # 统计总光强
    """for i,(det_x0, det_x1, det_y0, det_y1) in enumerate(detector_pos) : # 计算各个探测器区间内的光强占比
        if detector_mask is not None:
            Int[:, det_x0 : det_x1, det_y0 : det_y1]*=detector_mask[i]
        if detector_minus is not None:
            tmp=torch.zeros(len(detector_pos),Int.shape[0],Int.shape[1],Int.shape[2], device = device)
            tmp[i,:, det_x0 : det_x1, det_y0 : det_y1]=1.0
    if detector_minus is not None:
        tmp=torch.einsum("ibxy,i->bxy",tmp,detector_minus)
        Int=torch.sub(Int,tmp)
    for i,(det_x0, det_x1, det_y0, det_y1) in enumerate(detector_pos) :
        detectors_list[:, i] = Int[:, det_x0 : det_x1, det_y0 : det_y1].sum(dim=(1, 2))"""
    
    for i,(x,y) in enumerate(detector_pos) : # 计算各个探测器区间内的光强占比
        det_x0=torch.floor(x-detector_size/2).int()
        det_x1=torch.floor(x+detector_size/2).int()
        det_y0=torch.floor(y-detector_size/2).int()
        det_y1=torch.floor(y+detector_size/2).int()
        if detector_mask is not None:
            Int[:, det_x0 : det_x1+1, det_y0 : det_y1+1]*=detector_mask[i]
        if detector_minus is not None:
            tmp=torch.zeros(len(detector_pos),Int.shape[0],Int.shape[1],Int.shape[2], device = device)
            tmp[i,:, det_x0 : det_x1+1, det_y0 : det_y1+1]=1.0
    if detector_minus is not None:
        tmp=torch.einsum("ibxy,i->bxy",tmp,detector_minus)
        Int=torch.sub(Int,tmp)
    for i,(x,y) in enumerate(detector_pos) :
        det_x0=torch.floor(x-detector_size/2).int()
        det_x1=torch.floor(x+detector_size/2).int()
        det_y0=torch.floor(y-detector_size/2).int()
        det_y1=torch.floor(y+detector_size/2).int()
        detectors_list[:, i] = Int[:, det_x0 : det_x1, det_y0 : det_y1].sum(dim=(1, 2))+(Int[:,det_x1:det_x1+1,det_y0 : det_y1].sum(dim=(1, 2))-Int[:,det_x0:det_x0+1,det_y0 : det_y1].sum(dim=(1, 2)))*(x-detector_size/2-det_x0)+(Int[:,det_x0:det_x1,det_y1 : det_y1+1].sum(dim=(1, 2))-Int[:,det_x0:det_x1,det_y0 : det_y0+1].sum(dim=(1, 2)))*(detector_size/2-det_y0)

    total = detectors_list.sum(dim = 1)
    return Int, torch.einsum("ij,i->ij",detectors_list,torch.reciprocal(total))

# 指定生成的十个探测器的位置。
"""
detector_pos = [
    (240, 320, 400, 480),
    (240, 320, 720, 800),
    (560, 640, 870, 950),
    (860, 940, 720, 800),
    (860, 940, 400, 480),
    (560, 640, 250, 330)
]
"""

# 定义探测器模型基本参数
det_size = 60
det_pad_x = (PhaseMask[0] - 13*det_size)//2
det_pad_y = (PhaseMask[1] - 13*det_size)//2
detector_pos = set_det_pos(det_size, det_pad_y, det_pad_x)
"""
#FourDown Circle
detector_pos = [
    (570, 630, 200, 260),
    (840, 900, 300, 360),
    (940, 1000, 570, 630),
    (840, 900, 840, 900)
]
"""
"""
# FourUp
detector_pos = [
    (200, 260, 200, 260),
    (200, 260, 570, 630),
    (200, 260, 940, 1000),
    (570, 630, 940, 1000)
]
"""
"""
# FourUp LR Circle
detector_pos = [
    (300, 360, 300, 360),
    (200, 260, 570, 630),
    (300, 360, 840, 900),
    (570, 630, 940, 1000)
]
"""
"""
# Eight Circle
detector_pos = [
    (300, 360, 300, 360),
    (200, 260, 570, 630),
    (300, 360, 840, 900),
    (570, 630, 940, 1000),
    (570, 630, 200, 260),
    (840, 900, 300, 360),
    (940, 1000, 570, 630),
    (840, 900, 840, 900)
]
"""
"""
#six circle
detector_pos = [
    (270, 330, 770, 830),
    (570, 630, 940, 1000),
    (870, 930, 770, 830),
    (870, 930, 370, 430),
    (570, 630, 200, 260),
    (270, 330, 370, 430)
]
"""

"""# Circle Ten Size40 Near
detector_pos = [
    (580, 620, 200, 240),
    (803, 843, 273, 313),
    (941, 981, 463, 503),
    (941, 981, 697, 737),
    (803, 843, 887, 927),
    (580, 620, 960, 1000),
    (357, 397, 887, 927),
    (219, 259, 697, 737),
    (219, 259, 463, 503),
    (357, 397, 273, 313)
]"""

"""
# Circle Ten Size40 Far
detector_pos = [
    (180, 220, 710, 750),
    (333, 373, 920, 960),
    (580, 620, 1000, 1040),
    (827, 867, 920, 960),
    (980, 1020, 710, 750),
    (980, 1020, 450, 490),
    (827, 867, 240, 280),
    (580, 620, 160, 200),
    (333, 373, 240, 280),
    (180, 220, 450, 490)
]
"""
"""
# Circle Eight size 40
detector_pos = [
    (580, 620, 160, 200),
    (877, 917, 283, 323),
    (1000, 1040, 580, 620),
    (877, 917, 877, 917),
    (580, 620, 1000, 1040),
    (283, 323, 877, 917),
    (160, 200, 580, 620),
    (283, 323, 283, 323)
]
"""
"""
# Circle Eight size 60
detector_pos = [
    (300, 360, 300, 360),
    (200, 260, 570, 630),
    (300, 360, 840, 900),
    (570, 630, 940, 1000),
    (570, 630, 200, 260),
    (840, 900, 300, 360),
    (940, 1000, 570, 630),
    (840, 900, 840, 900)
]
"""
"""
# Circle Five Size40 Far
detector_pos = [
    (180, 220, 710, 750),
    
    (580, 620, 1000, 1040),
    
    (980, 1020, 710, 750),
    
    (827, 867, 240, 280),
    
    (333, 373, 240, 280)
]
"""
"""detector_pos = [
    (580, 620, 960, 1000),
    (580, 620, 200, 240)
]"""
"""# Eight minus 2 Circle
detector_pos = [
    (300, 360, 300, 360),
    (200, 260, 570, 630),
    (300, 360, 840, 900),
    (570, 630, 940, 1000),
    (840, 900, 840, 900),
    (940, 1000, 570, 630),
    #(840, 900, 300, 360),
    #(570, 630, 200, 260),
]"""
"""# Circle Ten Size40 Far
detector_pos = [
    (180, 220, 450, 490),
    (180, 220, 710, 750),
    (333, 373, 920, 960),
    #(580, 620, 1000, 1040),
    (827, 867, 920, 960),
    #(980, 1020, 710, 750),
    (980, 1020, 450, 490),
    (827, 867, 240, 280),
    
]"""
# Circle Ten Size40 Near
detector_pos = [
    #(580, 620, 200, 240),
    (803, 843, 273, 313),
    (941, 981, 463, 503),
    (941, 981, 697, 737),
    #(803, 843, 887, 927),
    (580, 620, 960, 1000),
    #(357, 397, 887, 927),
    (219, 259, 697, 737),
    #(219, 259, 463, 503),
    (357, 397, 273, 313)
]
detector_pos_xy=[]
for x0,x1,y0,y1 in detector_pos:
    detector_pos_xy.append(((x0+x1)/2,(y0+y1)/2))
labels_image_tensors=torch.zeros((len(detector_pos), PhaseMask[0], PhaseMask[1]), device = device, dtype = torch.float32)
for ind, pos in enumerate(detector_pos):
    pos_l, pos_r, pos_u, pos_d = pos
    labels_image_tensors[ind, pos_l:pos_r, pos_u:pos_d] = 1 # 设置探测器区域
    labels_image_tensors[ind] = labels_image_tensors[ind]/labels_image_tensors[ind].sum() # 归一化探测器层
det_ideal = labels_image_tensors.cpu().numpy().sum(axis = 0)

plt.rcParams["figure.figsize"] = (4, 4)
plt.imshow(det_ideal,'viridis') # 查看探测器层

## ***D2NN***

In [ ]:
class DNN(torch.nn.Module):
    """""""""""""""""""""
    phase only modulation
    """""""""""""""""""""
    def __init__(self, num_layers = 1, wl = wl, PhaseMask = PhaseMask, pixel_size = PIXEL_SIZE,
                 distance_between_layers = 0.2, distance_to_detectors = 0.2,num_classes=5):

        super(DNN, self).__init__()
        """
        # 初始化每层相位板的相位参数（0到1区间均匀分布）,并将其注册为可学习的Parameter。
        self.phase = [torch.nn.Parameter(torch.from_numpy(np.random.random(size=(PhaseMask[0], PhaseMask[1])).astype('float32'))) for _ in range(num_layers)]
        # 向网络中添加每层相位板的参数
        for i in range(num_layers):
            self.register_parameter("phase" + "_" + str(i), self.phase[i])
        """
        self.phase_mask = torch.nn.ParameterList([
            torch.nn.Parameter(torch.rand(PhaseMask, dtype=torch.float32)) for _ in range(num_layers)
        ])
        
        self.detector_mask = torch.nn.Parameter(torch.ones(num_classes))
        self.detector_minus = torch.nn.Parameter(torch.zeros(num_classes))
        self.detector_pos = torch.nn.Parameter(torch.tensor(detector_pos_xy))
        # 定义中间的衍射层
        self.diffractive_layers = torch.nn.ModuleList([Diffractive_Layer(wl, PhaseMask, pixel_size, distance_between_layers) for _ in range(num_layers)])
        # 定义最后的探测层
        self.last_diffractive_layer = Propagation_Layer(wl, PhaseMask, pixel_size, distance_to_detectors)
        self.sofmax = torch.nn.Softmax(dim=-1)

    # 计算多层衍射前向传播
    def forward(self, E):
        E = E.to(torch.cfloat)
        for index, layer in enumerate(self.diffractive_layers):
            temp = layer(E)
            # phase_values = 2 * np.pi * torch.sigmoid(self.phase_mask[index])
            phase_values = 2 * torch.pi * self.phase_mask[index]
            # 这里相当于加了一层sigmoid非线性激活，将相位控制在0到2pi
            #             constr_phase = 2*np.pi*torch.sigmoid(self.phase[index])
            modulation = torch.exp(1j*phase_values) #torch.cos(constr_phase)+1j*torch.sin(constr_phase)
            E = temp * modulation
        E = self.last_diffractive_layer(E)
        Int = torch.abs(E)**2
        #output = self.sofmax(detector_region(x_abs))
        Int,output = detector_region(Int,self.detector_mask,self.detector_minus,self.detector_pos)

        return output, Int

In [ ]:
model = DNN().to(device)
print(model)

In [ ]:
import time
currentDate = time.strftime("%Y%m%d_%H%M", time.localtime())
Date = time.strftime("%Y%m%d", time.localtime())
if not os.path.exists(f"weight/{Date}"):
    os.makedirs(f"weight/{Date}")
filename = f"{currentDate}_{type}_{dataset}.pth"

## ***Training***

In [ ]:
# 定义训练函数
def train(model, loss_function, optimizer, scheduler, trainloader, testloader, num_classes=5, epochs =5,  device=device, strict_accuracy_ratio=1,minus_mask_ratio=0,label_num=0,):
    if not os.path.exists(f"log/{Date}"):
        os.makedirs(f"log/{Date}")
    log_file = open(f"log/{Date}/log_{currentDate}.txt", "w")
    log_file.write(f"模型配置: Dataset: {dataset}\n loss: {loss_function}\n optimizer: {optimizer}\n scheduler: {scheduler}\n num_classes: {num_classes}\n epochs: {epochs}\n device: {device}\n strict_accuracy_ratio: {strict_accuracy_ratio}\n")
    log_file.write(f"Model saved as {filename}\n")
    train_loss_hist = []
    test_loss_hist = []
    train_acc_hist = []
    test_acc_hist = []
    best_acc = 0
    best_loss = 10000
    best_model_state = None
    squared_strict_accuracy_ratio=strict_accuracy_ratio**2
    if label_num>=2*num_classes: 
        num_classes+=1
    classes_num=num_classes
    #compute loss
    def compute_loss(images, labels,out_img):
        full_int_img = out_img.sum(axis=(1,2))
        normalized_out_img = out_img/full_int_img[:,None,None]
        # 修改损失计算逻辑
        #print(f"image:{images.shape}\nout_img:{out_img.shape}")

        batch_loss = torch.tensor(0.0, device=device)
        if label_num<2*classes_num: 
            if label_num!=0:
                print(f"[WARN]label num:{label_num} is less than class num:{classes_num}") 
            for i in range(images.size(0)):
                sample_label = labels[i].item()
                sample_img = normalized_out_img[i]
                possible_losses = []
                target_mask_weight=torch.Tensor([minus_mask_ratio for i in range(classes_num)])
                target_mask_weight[sample_label]=1
                target_mask_weight=target_mask_weight.to(device)
                target_mask = torch.einsum('c,chw->hw', target_mask_weight, labels_image_tensors)
                # 计算与每个有效目标的损失
                loss = loss_function(sample_img, target_mask)
                possible_losses.append(loss)
                if possible_losses:
                    min_loss_for_sample = torch.min(torch.stack(possible_losses))
                    batch_loss += min_loss_for_sample
        else:
            for i in range(images.size(0)):
                sample_label = labels[i].item()
                sample_img = normalized_out_img[i]
                possible_losses = []
                target_mask_weight=torch.Tensor([minus_mask_ratio for i in range(classes_num)])
                floor=((sample_label+1)*(classes_num-1))//(label_num)
                ceil=((sample_label+1)*(classes_num-1)+label_num-1)//(label_num)
                position=((sample_label+1)%((label_num)/(classes_num-1)))*((classes_num-1)/(label_num))
                if position==0:
                    target_mask_weight[floor]=1
                else:
                    target_mask_weight[floor]=1-position
                    target_mask_weight[ceil]=position
                # 计算与每个有效目标的损失
                target_mask_weight=target_mask_weight.to(device)
                target_mask = torch.einsum('c,chw->hw', target_mask_weight, labels_image_tensors)
                loss = loss_function(sample_img, target_mask)
                possible_losses.append(loss)
                if possible_losses:
                    min_loss_for_sample = torch.min(torch.stack(possible_losses))
                    batch_loss += min_loss_for_sample
        return batch_loss
    #compute accuracy
    def compute_acc(out_label, labels):
        base_intensity=1/num_classes
        # 找到最亮的探测器及其索引
        max_intensities, predicted = torch.max(out_label.data, 1) # 原始预测
        # 找到每个样本中第二亮探测器的亮度
        # 创建一个编码，将真实目标位置的值设为负无穷大
        non_target_mask_bool = F.one_hot(predicted, num_classes=num_classes).bool()
        # 找到除了目标之外的最大亮度
        second_brightest_intensity,nd_predicted = torch.max(out_label.masked_fill(non_target_mask_bool, -float('inf')), dim=1)
        is_correct = torch.zeros_like(labels, dtype=torch.bool)
        mask_map = {
            i: (predicted == i) for i in range(num_classes) 
        }
        max_intensities-=base_intensity
        second_brightest_intensity-=base_intensity
        if label_num<2*classes_num:
            for i in range(num_classes):
                mask = (labels == i)
                if mask.any():
                    is_correct[mask] = mask_map[i][mask] & (max_intensities[mask] >= second_brightest_intensity[mask]* strict_accuracy_ratio)
        else:
            for i in range(label_num):
                mask = (labels == i)
                if mask.any():
                    floor=((i+1)*(classes_num-1))//(label_num)
                    ceil=((i+1)*(classes_num-1)+label_num-1)//(label_num)

                    single_check=torch.logical_and(second_brightest_intensity[mask]<=0 , torch.abs(predicted[mask]-((i+1)*(classes_num-1)/(label_num)))<0.5)
                    if ceil==floor:
                        correct_pair=torch.logical_or((predicted[mask] == floor) , single_check)
                        intensity_check = max_intensities[mask]+base_intensity>= (second_brightest_intensity[mask]+base_intensity) * (strict_accuracy_ratio)
                    else:
                        # 检查最高的两个预测值是否为 floor 和 ceil（顺序不限）
                        non_target_mask_bool = torch.logical_or(non_target_mask_bool , F.one_hot(nd_predicted, num_classes=num_classes))
                        third_brightest_intensity, rd_predicted = torch.max(out_label.masked_fill(non_target_mask_bool, -float('inf')), dim=1)
                        #third_brightest_intensity-=base_intensity
                        correct_pair = torch.logical_or(torch.logical_or(
                            ((predicted[mask] == floor) & (nd_predicted[mask] == ceil)),
                            ((predicted[mask] == ceil) & (nd_predicted[mask] == floor))
                        ) , single_check)
                        # 检查强度是否满足严格准确率的要求
                        #intensity_check = (max_intensities[mask]**2 + second_brightest_intensity[mask]**2 >= (third_brightest_intensity[mask]**2) * (squared_strict_accuracy_ratio))
                        intensity_check = (max_intensities[mask] + second_brightest_intensity[mask] >= (third_brightest_intensity[mask]) * (strict_accuracy_ratio))
                    is_correct[mask] = correct_pair & intensity_check                    
        return is_correct
    # 训练循环
    for epoch in range(epochs):
        ep_train_loss = 0
        # 每个epoch开始时启动Batch_Normalization和Dropout。BN层能够用到每一批数据的均值和方差，Dropout随机取一部分网络连接来训练更新参数。
        model.train()
        correct = 0
        total = 0
        batch_size = 10
        for index, (images, labels) in enumerate(trainloader):
            optimizer.zero_grad() # 梯度清零
            images = images.to(device).float()

            with torch.no_grad(): # 增强操作本身不需要记录梯度
                images_aug = gpu_transform(images)
            # 注意：v2 输出通常保持维度 [B, C, H, W]，如果 Pad 需要 [B, H, W] 输入需 squeeze
            # 但 F.pad 其实支持 4D 输入，建议直接对 images_aug 操作
            if images_aug.shape[1] == 1:
                images_squeezed = images_aug.squeeze(1)
            else:
                images_squeezed = images_aug
                
            images = F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy)) # 二次Pad? 代码逻辑里似乎有两次
            labels = labels.to(device)
            # 模型输出
            out_label, out_img = model(images) # 得到预测各个探测器上的光强分布以及探测层光强分布
            # ---损失计算---
            total_loss = compute_loss(images=images, labels=labels,out_img=out_img)
            # --- 反向传播 ---
            total_loss.backward() # 反向传播
            optimizer.step() # 参数更新
            ep_train_loss += total_loss.item() # 累加更新本次epoch的损失

            # 新的准确率计算，不计算梯度
            with torch.no_grad():
                correct +=compute_acc(out_label=out_label, labels=labels).sum().item()
            total += labels.size(0) # 得到一个batch的标签总数
        # 计算平均训练损失和准确率
        avg_train_loss = ep_train_loss /len(trainloader.dataset)
        train_acc = correct /total
        train_loss_hist.append(avg_train_loss) # 计算平均损失
        train_acc_hist.append(train_acc) # 计算准确率
        # --- 验证循环 ---
        ep_test_loss = 0
        # 不启用Batch Normalization和Dropout。测试过程中要保证BN层的均值和方差不变，且利用到了所有网络连接，即不进行随机舍弃神经元。
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad(): # 停止梯度更新
            for index, (images, labels) in enumerate(testloader):
                images = images.to(device)
                labels = labels.to(device)
                images_aug = gpu_transform(images)
                # 注意：v2 输出通常保持维度 [B, C, H, W]，如果 Pad 需要 [B, H, W] 输入需 squeeze
                # 但 F.pad 其实支持 4D 输入，建议直接对 images_aug 操作
                if images_aug.shape[1] == 1:
                    images_squeezed = images_aug.squeeze(1)
                else:
                    images_squeezed = images_aug
                    
                images = F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy))
                out_label, out_img = model(images)
                #det_labels = F.one_hot(labels, num_classes=7).to(dtype=torch.float64)
                #det_labels = labels_image_tensors[labels]
                # --- 计算验证损失 ---
                total_val_loss = compute_loss(images=images, labels=labels,out_img=out_img)
                #loss = loss_function(out_label, det_labels)
                # 结束验证计算损失
                ep_test_loss += total_val_loss.item()
                # 计算准确率          
                correct += compute_acc(out_label=out_label,labels=labels).sum().item()
                total += labels.size(0)

        # 计算平均验证损失和准确率
        avg_test_loss = ep_test_loss/len(testloader)
        test_acc = correct/total
        test_loss_hist.append(avg_test_loss)
        test_acc_hist.append(test_acc)
        
        learning_rate = optimizer.param_groups[0]['lr']
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau): # 如果使用ReduceLROnPlateau调度器，则根据验证损失调整学习率
            scheduler.step(avg_test_loss)
        else:
            scheduler.step() # 学习率更新
            
        # 打印日志
        log_file.write(f"Epoch {epoch+1}/{epochs} | Learning rate: {learning_rate}\n")
        log_file.write(f"Train Loss: {avg_train_loss*1e4:.8f} | Train Acc: {train_acc:.8f}\n")
        log_file.write(f"Val Loss: {avg_test_loss*1e4:.8f} | Val Acc: {test_acc:.8f}\n")
        log_file.write(f"-"*50)
        log_file.write("\n")
        if (epoch+1) % 2 == 0 or epoch == epochs-1:
            print("-"*50)
            print(f"Epoch {epoch+1}/{epochs} | Learning rate: {learning_rate}")
            print(f"Train Loss: {avg_train_loss*1e4:.6f} | Train Acc: {train_acc:.6f}")
            print(f"Val Loss: {avg_test_loss*1e4:.6f} | Val Acc: {test_acc:.6f}")
            print("-"*50)
        
        # 如果最后一次训练的准确率大于之前最好的准确率，则将最后一次的模型保存为最佳模型。
        if test_acc>best_acc:
            best_acc = test_acc
            best_model_state = copy.deepcopy(model.state_dict())
            print(f" **Epoch {epoch+1}/{epochs} The newest Acc: {best_acc:.6f} (Already saved model state {time.strftime("%H:%M:%S", time.localtime())}) **")
            torch.save(best_model_state,"tmp.pth")

    # 加载最佳模型
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        best_model = model
        print(f"Already load the best model, Acc: {best_acc:.6f}")
    else:
        print("Unable to find the best, return the model with the last state")
        best_model = model
    log_file.write(f"Model saved as {filename}\n Best accuracy: {best_acc:.8f}\n")
    log_file.close() 
    return train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist, best_model, best_acc

In [ ]:
# 定义衍射模型基本参数
D2NNlayer_num = 1
wl = 532e-9
pixel_size = 8e-6
PhaseMask = [1200, 1200]
distance_between_layers = 0.175 #
distance_to_detectors = 0.225

# 训练的一些具体参数
num_classes=5
epochs=10
strict_accuracy_ratio = 1.0
# 定义模型，损失函数和优化器
model = DNN(D2NNlayer_num, wl, PhaseMask, pixel_size, distance_between_layers,num_classes=num_classes+1).to(device)
# Loss损失函数
#criterion = torch.nn.CrossEntropyLoss(reduction='sum').to(device)
criterion = torch.nn.MSELoss(reduction='sum').to(device)
#criterion = torch.nn.MSELoss(reduction='mean').to(device)
# Optimizer优化器
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
#optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
#optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
# Scheduler学习率调度器
#scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=0.25*epochs, gamma=0.1)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=4)#, verbose=True)

if not os.path.exists(f"HyperparametersLog/{Date}"):
    os.makedirs(f"HyperparametersLog/{Date}")
Hyperlog_file = open(f"HyperparametersLog/{Date}/Hyper_{currentDate}.txt", "w")
Hyperlog_file.write(f"预设超参数: D2NNlayer_num: {D2NNlayer_num}\n wl: {wl}\n pixel_size: {pixel_size}\n PhaseMask: {PhaseMask}\n distance_between_layers: {distance_between_layers}\n distance_to_detectors: {distance_to_detectors}\n num_classes: {num_classes}\n epochs: {epochs}\n strict_accuracy_ratio: {strict_accuracy_ratio}\n BATCH_SIZE: {BATCH_SIZE}\n")
Hyperlog_file.close()
if os.path.exists("tmp.pth"):
    model.load_state_dict(torch.load("tmp.pth",weights_only=False))
    best_model=model
    print("Load tmp.pth")

In [ ]:
# 正式开启训练
train_loss_hist, train_acc_hist, test_loss_hist, test_acc_hist, best_model, best_acc = train(model,
                                criterion,optimizer, scheduler, train_dataloader, val_dataloader, num_classes=num_classes, epochs=epochs,  device=device, strict_accuracy_ratio=strict_accuracy_ratio,minus_mask_ratio=0.0,label_num=20)

## ***Curve Plotting***

In [ ]:
import matplotlib.pyplot as plt
import os

def plot_training_curves(train_loss_hist, val_loss_hist,item_name=None ,save_path=None):
    plt.figure(figsize=(14, 6))
    plt.subplot(1, 2, 1)
    epochs = range(1, len(train_loss_hist) + 1)
    plt.plot(epochs, train_loss_hist, 'o-', label=f'Training {item_name}', color='royalblue')
    plt.title(f'Train {item_name} History', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(f'{item_name}', fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.subplot(1, 2, 2)
    epochs = range(1, len(train_loss_hist) + 1)
    plt.plot(epochs, val_loss_hist, 's--', label=f'Validation {item_name}', color='darkorange')
    plt.title(f'Val {item_name} History', fontsize=16)
    plt.xlabel('Epochs', fontsize=12)
    plt.ylabel(item_name, fontsize=12)
    plt.legend(fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.tight_layout()
    if not os.path.exists(f"{save_path}"):
        os.makedirs(f"{save_path}")
    if save_path:
        saveContent = f"{save_path}/{dataset}_Loss.png"
        plt.savefig(saveContent, dpi=1200, bbox_inches='tight')
        print(f"Saved plot to {saveContent}")
    plt.show()

print("正在绘制训练损失图……")
plot_training_curves(train_loss_hist, test_loss_hist,"Loss", f"results/{type}/Loss")
plot_training_curves(train_acc_hist, test_acc_hist,"Acc", f"results/{type}/Acc")


## ***Saving***

In [ ]:

torch.save(best_model, f"weight/{Date}/{filename}")
print(f"Model saved as {filename}")

In [ ]:
# 释放显存
torch.cuda.empty_cache()

## ***Loading***

In [ ]:
#torch.serialization.add_safe_globals(torch.serialization.get_unsafe_globals_in_checkpoint(torch.load(f'weight/{Date}/{filename}',weights_only=False)))
torch.serialization.clear_safe_globals()
#torch.serialization.add_safe_globals()
#print(torch.serialization.get_unsafe_globals_in_checkpoint(f'weight/{Date}/{filename}'))
model = torch.load(f'weight/{Date}/{filename}',weights_only=False)
#model = torch.load(f'weight/20250619/20250619_20250619_0718.pth')

## ***Data Analysis***

In [ ]:
# 查看
plt.rcParams["figure.figsize"] = (15, 4.5)
def visualize(image, label):
    image_padded = F.pad(image, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy))
    image_E = torch.sqrt(image_padded)
    out = model(image_E.to(device))
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, constrained_layout=True)
    ax1.grid(False)
    ax1.imshow(image.squeeze(), interpolation='none', cmap='viridis')
    ax1.set_title(f'Input image with\n total intensity {image.squeeze().sum():.2f}')
    output_image = torch.abs(out[1].squeeze()).detach().cpu()
    ax2.grid(False)
    ax2.imshow(output_image*det_ideal, interpolation='none', cmap='viridis')
    ax2.set_title(f'Output image with\n total intensity {output_image.squeeze().sum():.2f}')
    dist = out[0].squeeze().detach().cpu()
    ax3.bar(range(len(dist)), dist)
    fig.suptitle("label={}".format(label))
    plt.grid(False)
    plt.show()
    print(image.shape)
    #print(images_padded.shape)
    print(image_E.shape)


def mask_visualiztion(Date, filename, save_path=None):
    if not os.path.exists(f'post/{Date}/{filename}'):
        os.makedirs(f'post/{Date}/{filename}')
    for ind, mask in enumerate(model.phase_mask):
        ## MATRIX CONSERVATION
        original_matrix = mask.detach().cpu().numpy()
        original_df = pd.DataFrame(original_matrix)
        original_df.to_csv(f'post/{Date}/{currentDate}_{type}_{dataset}.pth/original_mask.csv', index=False, header=False)
        
        original_matrix[original_matrix > 1] -= 1
        original_matrix[original_matrix < 0] += 1
        original_processed_df = pd.DataFrame(original_matrix) #原来的本身的数据
        original_processed_df.to_csv(f'post/{Date}/{currentDate}_{type}_{dataset}.pth/original_mask_processed.csv', index=False, header=False)
        
        print("保存完成！")
        ## VISUALIZATION
        plt.imshow((original_matrix / np.max(original_matrix)) * 1023, interpolation = 'none', cmap='viridis')
        #plt.imshow(mask.detach().cpu()*180/np.pi, interpolation='none')
        plt.title(f'Mask of layer {ind+1}')
        plt.colorbar()
        plt.grid(False)
        if not os.path.exists(f"{save_path}"):
            os.makedirs(f"{save_path}")
        if save_path:
            saveContent = f"{save_path}/{dataset}_Mask.png"
            plt.savefig(saveContent, dpi=1200, bbox_inches='tight')
        print(f"Saved plot to {saveContent}")
        plt.show()

In [ ]:
mask_visualiztion(Date, filename, f"results/{type}/PhaseMask")

In [ ]:
"""
rand_ind = np.random.choice(range(len(val_dataset)), size=10, replace=False)
for ind in rand_ind:
    visualize(val_dataset[ind][0], val_dataset[ind][1])
 """

## ***Confusion Matrix***

In [ ]:
"""import seaborn as sns
from sklearn.metrics import confusion_matrix
def plot_confusion_matrix(model, dataloader, device, classes_num, save_path=None):
    model.eval()
    all_labels = []
    all_preds = []
    classesname = [f'{i}' for i in range(classes_num)]
    print("计算混淆矩阵……")
    with torch.no_grad(): # 停止梯度更新
        for i, (images, labels) in enumerate(dataloader):
            images = images.to(device).float()
            with torch.no_grad(): # 增强操作本身不需要记录梯度
                images_aug = gpu_transform(images)
            # 注意：v2 输出通常保持维度 [B, C, H, W]，如果 Pad 需要 [B, H, W] 输入需 squeeze
            # 但 F.pad 其实支持 4D 输入，建议直接对 images_aug 操作
            if images_aug.shape[1] == 1:
                images_squeezed = images_aug.squeeze(1)
            else:
                images_squeezed = images_aug
                
            #images = F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy))

            images_E = torch.sqrt(F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy)))
            labels = labels.to(device)

            out_labels, out_images = model(images_E)
            _, predicted = torch.max(out_labels, 1)

            all_labels.append(labels.cpu().numpy())
            all_preds.append(predicted.cpu().numpy())

    all_labels = np.concatenate(all_labels)
    all_preds = np.concatenate(all_preds)

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm, annot=True, annot_kws={"size": 40}, fmt='.2f', cmap='Blues', xticklabels=classesname, yticklabels=classesname)
    plt.title('Confusion  Matrix', fontsize=28)
    plt.xlabel('Predicted  Label', fontsize=26)
    plt.ylabel('True  Label', fontsize=26)
    plt.xticks(fontsize=24)
    plt.yticks(fontsize=24)
    if not os.path.exists(f"{save_path}"):
        os.makedirs(f"{save_path}")
    # 如果提供了保存路径，则保存图像
    if save_path:
        plt.savefig(f"{save_path}/{dataset}_Confusion.png",  dpi=1200, bbox_inches='tight')  # 高分辨率保存，避免裁剪
        print(f"混淆矩阵已保存至: {save_path}")

    plt.show()   # 显示图像


confusion_matrix_result = plot_confusion_matrix(model, val_dataloader, device, 2, f"results/{type}/ConfusionMatrix")"""

In [ ]:
def confusion_matrix(predicted, nd_predicted,max_intensities,second_max_intensities,labels, conf_matrix,classes_num,label_num=None):
    if label_num==None:
        label_num=classes_num
    base_intensity=1/(classes_num+1)
    for p,ndp,mi,ndi, i in zip(predicted,nd_predicted, max_intensities,second_max_intensities, labels):
        if ndi<=base_intensity:
            if 0<p<classes_num:
                conf_matrix[p-1, i] += 0.5
                conf_matrix[p, i] += 0.5
            else:
                conf_matrix[min(p,classes_num-1), i] += 1
        #elif (p==floor and ndp==ceil) or (p==ceil and ndp==floor):
        elif abs(p-ndp)==1:
            conf_matrix[min(p,ndp), i] += 1
        elif p==classes_num or ndp==classes_num:
            conf_matrix[classes_num-1, i] += 1
        else:
            #continue
            conf_matrix[min(p,classes_num-1), i] += 1*(mi/(mi+ndi))
            conf_matrix[min(ndp,classes_num-1), i] += 1*(ndi/(mi+ndi))
    return conf_matrix   
#首先定义一个 分类数*分类数 的空混淆矩阵
matrix_label_num=20
conf_matrix = torch.zeros(num_classes,matrix_label_num ).to(device)
# 使用torch.no_grad()可以显著降低测试用例的GPU占用
with torch.no_grad(): # 停止梯度更新
    for i, (images, labels) in enumerate(val_dataloader):
        images = images.to(device)
        images_aug = gpu_transform(images)
        # 注意：v2 输出通常保持维度 [B, C, H, W]，如果 Pad 需要 [B, H, W] 输入需 squeeze
        # 但 F.pad 其实支持 4D 输入，建议直接对 images_aug 操作
        if images_aug.shape[1] == 1:
            images_squeezed = images_aug.squeeze(1)
        else:
            images_squeezed = images_aug
        images = torch.sqrt(F.pad(images_squeezed, pad=(PADDINGx, PADDINGx, PADDINGy, PADDINGy)))
        labels = labels.to(device)

        out_labels, out_images = model(images)
        max_intensities, predicted = torch.max(out_labels.data, 1)
        non_target_mask_bool = F.one_hot(predicted, num_classes=num_classes+1).bool()
        # 找到除了目标之外的最大亮度
        second_brightest_intensity,nd_predicted = torch.max(out_labels.masked_fill(non_target_mask_bool, -float('inf')), dim=1)
        #记录混淆矩阵参数
        conf_matrix = confusion_matrix(predicted,nd_predicted,max_intensities,second_brightest_intensity, labels, conf_matrix,num_classes,matrix_label_num)
conf_matrix = conf_matrix.cpu()


In [ ]:
conf_matrix = np.array(conf_matrix)# 将混淆矩阵从gpu转到cpu再转到np
corrects = conf_matrix.diagonal(offset = 0)#抽取对角线的每种分类的识别正确个数
per_classes = conf_matrix.sum(axis = 1)#抽取每个分类数据总的测试条数

print("混淆矩阵总元素个数：{},测试集总个数:{}".format(int(np.sum(conf_matrix)), BATCH_SIZE*len(val_dataloader)))
np.set_printoptions(suppress=True)
print(conf_matrix)

# 获取每种label的识别准确率
percent = [rate*100 for rate in corrects/per_classes]
per_classes = list(per_classes)
corrects = list(corrects)

print("每种标签总个数：{}".format('  '.join(['{:.0f}'.format(i) for i in per_classes])))
print("每种标签预测正确的个数：{}".format('  '.join(['{:.0f}'.format(i) for i in corrects])))
print("每种标签的识别准确率为：{}".format('%  '.join(['{:.1f}'.format(i) for i in percent])), end='%')

In [ ]:
# 绘制混淆矩阵
"""
label_ticks = list()
for i in range(num_classes):
    label_ticks = label_ticks + ['{}'.format(classes[i])]
"""
# 显示数据
plt.imshow(conf_matrix, cmap=plt.cm.Blues)
thresh = conf_matrix.max() / 2	#数值颜色阈值，如果数值超过这个，就颜色加深。
for i in range(20):
    for j in range(num_classes):
        info = int(conf_matrix[j, i])
        #info="{:.2f}".format(float(conf_matrix[j, i]))
        plt.text(i, j, info,
                 verticalalignment='center',
                 horizontalalignment='center',
                 color="white" if info > thresh else "black")
plt.show()
print(model.detector_mask)
print(model.detector_minus)
for ind, pos in enumerate(model.detector_pos):
    x,y = pos
    detector_size=60
    pos_l=torch.floor(x-detector_size/2).int()
    pos_r=torch.floor(x+detector_size/2).int()
    pos_u=torch.floor(y-detector_size/2).int()
    pos_d=torch.floor(y+detector_size/2).int()
    labels_image_tensors[ind, pos_l:pos_r, pos_u:pos_d] = 1 # 设置探测器区域
    labels_image_tensors[ind] = labels_image_tensors[ind]/labels_image_tensors[ind].sum() # 归一化探测器层
det_ideal = labels_image_tensors.cpu().numpy().sum(axis = 0)

plt.rcParams["figure.figsize"] = (4, 4)
plt.imshow(det_ideal,'viridis') # 查看探测器层

train()的label_num 参数传递需要自动化

模型更改:加入探测层每个探测器前加一个衰减系数或者阈值作为可训练参数,以及探测器的位置也是可训练参数